##Title & Overview
# ML Research Architect (Multi-Agent System)

This notebook builds a **multi-agent AI system** using CrewAI to analyze machine learning research papers.

## Objective
Transform a research topic into a **structured theoretical foundation** including:
- Motivation
- Model Architecture
- Mathematical Equations (LaTeX)
- Tensor Dimensions (Tables)

---

## System Design

We use:
- **Agent-Based Architecture**
- **State Management (Pydantic)**
- **CrewAI Flow (Sequential Execution)**
- **Gradio UI for Interaction**

---

## Workflow

1. User inputs a research topic  
2. Research Agent analyzes the paper  
3. Output is formatted into:
   - LaTeX equations
   - Markdown tables  
4. Display results in UI  

---

This is a **real-world AI system prototype** for research automation.

##Step 1:Install Dependencies

In [ ]:
# 1. Install dependencies
!pip install crewai[tools] -q
!pip install gradio -q

##Step 2: Imports
We import:
- `gradio` → UI
- `pydantic` → state management
- `crewai` → agent orchestration

In [ ]:
import gradio as gr
from typing import List
from pydantic import BaseModel
from crewai import Agent, Task, Crew, LLM
from crewai.flow.flow import Flow, listen, start

##Step 3: State Definition
We define a structured state using **Pydantic**.

### Why State?
- Maintains data across steps
- Ensures structured flow
- Makes system scalable

### Fields:
- `paper_topic` → Input topic
- `math_foundation` → Final output

In [ ]:
class MLArchitectState(BaseModel):
    paper_topic: str = ""
    math_foundation: str = ""

##Step 4: LLM Setup
We configure a lightweight model for:
- High accuracy (low temperature)
- Structured outputs
- Technical reasoning

In [ ]:
custom_llm = LLM(
    model="gpt-4.1-nano",
    temperature=0.2,
    base_url="YOUR_BASE_URL",
    api_key="YOUR_API_KEY" # Replace with your actual key
)

##Step 5: Flow Explanation
## ML Architect Flow

We use CrewAI Flow to define execution steps.

### Step:
Research Phase

Agent Responsibilities:
- Extract architecture
- Derive equations
- Present tensor shapes

---

### Output Format Rules:
- LaTeX equations using $$ $$
- Markdown tables
- Proper spacing for readability

In [ ]:
class MLArchitectFlow(Flow[MLArchitectState]):

    @start()
    def research_phase(self):
        topic = self.state.paper_topic
        researcher = Agent(
            role="Senior ML Researcher",
            goal=f"Extract the core architectural logic and mathematical proofs for {topic}",
            backstory="An elite academic researcher who specializes in translating complex ML papers into clear, structured theoretical foundations.",
            llm=custom_llm,
            verbose=True
        )

        # Focused strictly on theoretical context
        task = Task(
        description=(
            f"Deep dive into {topic}. Provide a breakdown of: "
            "1. Motivation, 2. Architecture, 3. Math, 4. Tensor Dimensions."
            "\n\n"
            "### STRICT FORMATTING RULES:\n"
            "1. MATH: Every single equation MUST be on its own line and wrapped in double dollar signs. "
            "Example:\n\n"
            "$$ H^{(l)} = \text{LayerNorm}(H^{(l-1)} + \text{MHA}(H^{(l-1)})) $$\n\n"
            "2. TABLES: You MUST use a standard Markdown table with pipes '|'. "
            "Example:\n"
            "| Component | Input | Output | Description |\n"
            "| :--- | :--- | :--- | :--- |\n"
            "| Encoder | $[B, T, D]$ | $[B, T, D]$ | Description |\n\n"
            "3. SPACING: Put a blank empty line before and after EVERY table and EVERY equation block."
        ),
        expected_output="A clean research dossier where math and tables are perfectly rendered with proper spacing.",
        agent=researcher
    )

        result = Crew(agents=[researcher], tasks=[task]).kickoff()
        self.state.math_foundation = str(result)
        return str(result)

##Step 6: App Logic
## Execution Wrapper

This function:
- Takes user input
- Runs the flow
- Handles errors
- Returns output

In [ ]:
def run_app(topic):
    if not topic:
        return "⚠️ Please enter a topic", ""

    flow = MLArchitectFlow()
    flow.state.paper_topic = topic

    try:
        flow.kickoff()
        # Return status and the single research output
        return "✅ Research Dossier Generated", flow.state.math_foundation
    except Exception as e:
        return f"❌ Error: {str(e)}", "Please check your API key or connection."


##Step 7: UI Explanation
## Gradio Interface

We build a clean UI:

### Features:
- Topic input box
- Status indicator
- Large output display

---

Designed for:
- Research readability
- LaTeX rendering
- Clean structure

In [ ]:
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🔬 ML Research Architect")
    gr.Markdown("Extracting the mathematical and structural context from complex AI papers.")

    with gr.Row():
        with gr.Column(scale=4):
            input_text = gr.Textbox(
                label="Research Topic / Paper Name",
                placeholder="e.g., Attention Is All You Need, Whisper ASR, LoRA...",
                lines=1
            )
        with gr.Column(scale=1):
            btn = gr.Button("Analyze Architecture", variant="primary")

    status = gr.Markdown("### Status: *Waiting for Input*")

    with gr.Row():
        # Large, single-column output for better readability of LaTeX and long-form text
        out_math = gr.Markdown(label="Theoretical Foundation & Context")

    btn.click(
        fn=run_app,
        inputs=input_text,
        outputs=[status, out_math]
    )

demo.launch(share=True, debug=True)